# WhisperTranscribe — Colab GPU 実行用 Notebook

デスクトップアプリ [`WhisperTranscribe`](https://github.com/NamakeroBK/WhisperTranscribe) で前処理した 16kHz mono WAV を、Colab の **無料 T4 GPU** で高速に文字起こしします。

## 使い方（おおまかな流れ）

1. **GPU を有効化**: 上部メニュー → ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択
2. **Google Drive を準備**: マイドライブに以下フォルダを作っておく
   - `/MyDrive/WhisperTranscribe/inbox/`  ← ここに WAV をアップロード
   - `/MyDrive/WhisperTranscribe/outbox/` ← 結果がここに保存される
3. **設定セルを編集**（必要なら）: モデル名・言語・initial_prompt
4. **すべてのセルを実行**: メニュー → ランタイム → すべてのセルを実行 (Ctrl+F9)
5. **結果を取得**: `/MyDrive/WhisperTranscribe/outbox/` から `.txt`/`.srt`/`.vtt` をダウンロード

> **アプリ側のCPU推論より約 5〜15 倍高速**になります。

## 1. GPU 確認

In [ ]:
!nvidia-smi

## 2. ライブラリのインストール

[`faster-whisper`](https://github.com/SYSTRAN/faster-whisper) を使います（CTranslate2 ベースで標準の whisper より 2〜4 倍高速）。

In [ ]:
!pip install -q faster-whisper==1.0.3

## 3. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
INBOX  = '/content/drive/MyDrive/WhisperTranscribe/inbox'
OUTBOX = '/content/drive/MyDrive/WhisperTranscribe/outbox'
os.makedirs(INBOX, exist_ok=True)
os.makedirs(OUTBOX, exist_ok=True)

files = sorted([f for f in os.listdir(INBOX) if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac'))], key=lambda f: os.path.getmtime(os.path.join(INBOX, f)))
if not files:
    raise FileNotFoundError(f'{INBOX} に音声ファイルがありません。WAV/MP3/M4A/FLAC のいずれかをアップロードしてください。')
print(f'inbox 内のファイル ({len(files)} 件):')
for f in files: print(' ', f)
print(f'\n→ 最新ファイルを処理対象にします: {files[-1]}')

## 4. 設定（必要に応じて編集）

In [ ]:
# 処理対象ファイル名 (inbox 内・拡張子込み)。空文字なら inbox の最新ファイルを自動選択。
TARGET_FILE = ''

# モデル: tiny / base / small / medium / large-v3 / large-v3-turbo
MODEL = 'large-v3-turbo'

# 言語: 'ja' / 'en' / 'auto' など (faster-whisper の言語コード)
LANGUAGE = 'ja'

# 初期プロンプト: 「指示」ではなく「模範的な書き起こし例」を入れる (224 トークン上限)
# 句読点付き完成文 2〜3 文 + 固有名詞 10〜15 語が目安。空文字なら未指定。
INITIAL_PROMPT = ''

# ビームサーチ幅: 5 (標準) / 10 (高品質、約2倍遅い)
BEAM_SIZE = 5

# 計算精度: 'float16' (T4 GPU 推奨) / 'int8_float16' (省メモリ) / 'float32' (CPU)
COMPUTE_TYPE = 'float16'

if not TARGET_FILE:
    TARGET_FILE = files[-1]
INPUT_PATH = os.path.join(INBOX, TARGET_FILE)
assert os.path.exists(INPUT_PATH), f'{INPUT_PATH} が見つかりません'
print(f'入力 : {INPUT_PATH}')
print(f'モデル: {MODEL} / 言語: {LANGUAGE} / beam: {BEAM_SIZE} / dtype: {COMPUTE_TYPE}')
if INITIAL_PROMPT: print(f'初期プロンプト ({len(INITIAL_PROMPT)} 文字): {INITIAL_PROMPT[:80]}...')
else: print('初期プロンプト: なし')

## 5. 文字起こし実行

In [ ]:
import time
from faster_whisper import WhisperModel

t0 = time.time()
print(f'モデル読み込み中: {MODEL} ...')
model = WhisperModel(MODEL, device='cuda', compute_type=COMPUTE_TYPE)
print(f'  読み込み完了 ({time.time() - t0:.1f}s)')

t1 = time.time()
print(f'文字起こし開始: {INPUT_PATH}')
segments_iter, info = model.transcribe(
    INPUT_PATH,
    language=None if LANGUAGE == 'auto' else LANGUAGE,
    beam_size=BEAM_SIZE,
    temperature=0.0,
    condition_on_previous_text=False,  # 幻覚抑制
    initial_prompt=INITIAL_PROMPT or None,
    vad_filter=True,                   # 無音区間スキップ
    vad_parameters=dict(min_silence_duration_ms=1000, threshold=0.6),
)
print(f'  検出言語: {info.language} (確度 {info.language_probability:.2%})')
print(f'  音声長さ : {info.duration:.1f}s')

# ストリーミングをリスト化しつつ進捗表示
segments = []
last_log = 0
for s in segments_iter:
    segments.append(s)
    if s.end - last_log > 30 or len(segments) % 20 == 0:
        print(f'  ... {s.end:.1f}s / {info.duration:.1f}s  ({s.end / info.duration * 100:.1f}%)')
        last_log = s.end

elapsed = time.time() - t1
print(f'\n完了: {len(segments)} セグメント / 推論 {elapsed:.1f}s / 倍速比 {info.duration / elapsed:.1f}x')

## 6. 結果を `outbox` に書き出し

In [ ]:
base = os.path.splitext(TARGET_FILE)[0]
out_base = os.path.join(OUTBOX, base + '_whisper')

def _fmt(t, sep=','):
    h = int(t // 3600); m = int((t % 3600) // 60); s = t % 60
    return f'{h:02d}:{m:02d}:{int(s):02d}{sep}{int(round((s - int(s)) * 1000)):03d}'

# TXT
with open(out_base + '.txt', 'w', encoding='utf-8') as f:
    for s in segments:
        f.write(s.text.strip() + '\n')

# SRT
with open(out_base + '.srt', 'w', encoding='utf-8') as f:
    for i, s in enumerate(segments, 1):
        f.write(f'{i}\n{_fmt(s.start)} --> {_fmt(s.end)}\n{s.text.strip()}\n\n')

# VTT
with open(out_base + '.vtt', 'w', encoding='utf-8') as f:
    f.write('WEBVTT\n\n')
    for s in segments:
        f.write(f'{_fmt(s.start, ".")} --> {_fmt(s.end, ".")}\n{s.text.strip()}\n\n')

print('書き出し完了:')
print(f'  {out_base}.txt')
print(f'  {out_base}.srt')
print(f'  {out_base}.vtt')
print('\nGoogle Drive の outbox フォルダから取得してください。')
print('または下のセルでローカル PC へ直接ダウンロードできます。')

## 7. (オプション) ローカル PC へ直接ダウンロード

In [ ]:
from google.colab import files
files.download(out_base + '.srt')
files.download(out_base + '.txt')
# files.download(out_base + '.vtt')  # 必要なら有効化

## 8. (オプション) 先頭セグメントのプレビュー

In [ ]:
for s in segments[:20]:
    print(f'[{_fmt(s.start, ".")} -> {_fmt(s.end, ".")}] {s.text.strip()}')
if len(segments) > 20:
    print(f'... 他 {len(segments) - 20} セグメント')